In [1]:
import os, json, math, numpy as np, torch
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForCausalLM
from trl import SFTTrainer, SFTConfig
from peft import LoraConfig, TaskType
import pandas as pd

In [2]:
MISTRAL_NAME = "mistralai/Mistral-7B-v0.1"
CAI_DIR = "sft_data_icai"
MODELS_DIR = "models_icai"
SFT_MODEL_OUT = "mistral-base_v01-sft"
SFT_CRITIQUE_AND_REVISION = os.path.join(CAI_DIR, "sft_revision_and_critique.jsonl")
OUTPUT_DIR = os.path.join(MODELS_DIR, SFT_MODEL_OUT)
TRAINING_LOGS = os.path.join(MODELS_DIR, f"training_logs_full_{SFT_MODEL_OUT}.csv")
SEED = 42
TRAIN_FRAC = 0.9
MAX_SEQ_LEN = 2048

os.makedirs(OUTPUT_DIR, exist_ok=True)
rng = np.random.default_rng(SEED)

In [3]:
raw = [json.loads(l) for l in open(SFT_CRITIQUE_AND_REVISION, "r", encoding="utf-8") if l.strip()]
pairs = [{
    "messages": [
        {"role": "user", "content": r["prompt"].strip()},
        {"role": "assistant", "content": r["revisions"][0]["revised"].strip()},
    ]
} for r in raw]

seen, clean = set(), []
for ex in pairs:
    k = (ex["messages"][0]["content"], ex["messages"][1]["content"])
    if k in seen: continue
    seen.add(k); clean.append(ex)
pairs = clean
print(f"Total SFT pairs: {len(pairs)}")

idx = np.arange(len(pairs)); rng.shuffle(idx)
cut = math.floor(TRAIN_FRAC * len(idx))
train_items = [pairs[i] for i in idx[:cut]]
val_items   = [pairs[i] for i in idx[cut:]]
sft_ds = DatasetDict(
    train=Dataset.from_list(train_items),
    validation=Dataset.from_list(val_items)
)
print(f"Train: {len(train_items)}  |  Val: {len(val_items)}")

Total SFT pairs: 1000
Train: 900  |  Val: 100


In [4]:
tok = AutoTokenizer.from_pretrained(MISTRAL_NAME, use_fast=True)
if tok.pad_token_id is None:
    tok.pad_token = tok.eos_token  

tok.chat_template = """{% for message in messages %}
{% if message['role'] == 'user' %}
{{ '<|user|>\\n' + message['content'] + eos_token }}
{% elif message['role'] == 'system' %}
{{ '<|system|>\\n' + message['content'] + eos_token }}
{% elif message['role'] == 'assistant' %}
{{ '<|assistant|>\\n'  + message['content'] + eos_token }}
{% endif %}
{% if loop.last and add_generation_prompt %}
{{ '<|assistant|>' }}
{% endif %}
{% endfor %}"""

base_model = AutoModelForCausalLM.from_pretrained(
    MISTRAL_NAME,
    torch_dtype=(torch.bfloat16 if torch.cuda.is_available() else torch.float32),
    device_map="auto",
)

lora_cfg = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)


sft_cfg = SFTConfig(
    output_dir=OUTPUT_DIR,
    do_train=True,
    do_eval=True,
    logging_strategy="steps",     
    logging_steps=10,  
    eval_strategy="steps",
    eval_steps=10,
    bf16=torch.cuda.is_available(),
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    learning_rate=2e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    per_device_train_batch_size=1,     
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,  
    num_train_epochs=1,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=1,
    seed=42,
    remove_unused_columns=True,
    packing=False,                    
    padding_free=False,
)


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

def chat_template_formatting(example: Dict[str, Any]) -> str:
    return tok.apply_chat_template(
        example["messages"],
        add_generation_prompt=False,
        tokenize=False
    )

In [5]:
trainer = SFTTrainer(
    model=base_model,
    train_dataset=sft_ds["train"],
    eval_dataset=sft_ds["validation"],
    args=sft_cfg,
    peft_config=lora_cfg,
    processing_class=tok,
)

trainer.train()
trainer.save_model(OUTPUT_DIR)
tok.save_pretrained(OUTPUT_DIR)
print("LoRA adapter saved to:", OUTPUT_DIR)

Tokenizing train dataset:   0%|          | 0/900 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/900 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

The model is already on multiple devices. Skipping the move to device specified in `args`.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
10,1.657800,1.360123,1.498635,21337.000000,0.649540
20,1.131700,1.071217,1.123175,42036.000000,0.713744
30,1.041500,1.021505,1.079323,64646.000000,0.720349
40,0.962500,1.001768,1.011873,87829.000000,0.722846
50,0.981600,0.995521,1.005197,110241.000000,0.724697


LoRA adapter saved to: models_icai/mistral-base_v01-sft


In [6]:
logs = pd.DataFrame(trainer.state.log_history)
logs.to_csv(TRAINING_LOGS, index=False)